<div style="background:#E9FFF6; color:#440404; padding:8px; border-radius: 4px; text-align: center; font-weight: 500;">IFN619 - Data Analytics for Strategic Decision Makers</div>

# IFN619 :: B3-Semi/unstructured Analytics (Part A)

For this session, the focus will be on analysis of unstructured text. However, the thinking required is similar to approaches to analysing images, video, sound and other unstructured data. Primarily, the analysis is based on the notion that there are useful patterns in the unstructured data which can be obtained computationally.

Most of the time, working with semi-structured or unstructured data involves a process of incrementally structuring it to the point where we can obtain meaningful information. Algorithms like topic modelling algorithms (which we will look at in Part B) allow more complex analysis of semi/unstructured data. For now, we will look at some basic approaches.

In [ ]:
# Import the necessary libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import pandas as pd
import json
import random

### Basic incremental structuring

Before using the imported Python libraries, let's look at how we can work with unstructured text data by taking a simple (unstructured) string of characters and modify it to be more structured.

In [ ]:
# selected from David Bohm's 1990 paper: "A new theory of the relationship of mind and matter" (p.281)

bohm_text = "Thus, for example, when we read a printed page, we do not assimilate the substance of the paper, but only the forms of the letters, and it is these forms which give rise to an information content in the reader which is manifested actively in his or her subsequent activities. A similar mind-like quality of matter reveals itself strongly at the quantum level, in the sense that the form of the wave function manifests itself in the movements of the particles. This quality does not, however, appear to a significant extent at the level at which classical physics is a valid approximation."

# show the contents of the variable
???

We can split the string into separate sentences using the `split()` function.

In [ ]:
# Use an appropriate character to split text into sentences
bohm_sents = bohm_text.split(???)
bohm_sents

We can then process each sentence to extract words, adding them to a list

In [ ]:
bohm_words = []
for sent in bohm_sents:
    # Use an appropriate character to split sentences into words
    sent_words = sent.split(???)
    for word in sent_words:
        bohm_words.append(word)
        
bohm_words

We can clean up this list by removing empty elements, dropping commas, and making all words lowercase

In [ ]:
# Only keep words that are not '' (empty string), make them lower case and remove any comma characters
bohm_clean_words = [w.lower().replace(',','') for w in bohm_words if w!='']
bohm_clean_words

Finally, we can count how many of each word we have to get an indication of David Bohm's vocabulary used in this quote

In [ ]:
bohm_vocab = {}
for word in bohm_clean_words:
    bohm_vocab[word] = bohm_vocab.get(word, 0) + 1

# Sort vocab by count (value)
bohm_vocab_sorted = dict(sorted(bohm_vocab.items(), key=lambda item: item[1]))

# Show the final dictionary
???

What do you notice about the words in the vocabulary? 
Which words are meaningful? Which are not?

How do you think we could extract the meaning from the unstructured text?

---

## Using vectorizors to work with unstructured text

#### Accessing the data via The Guardian API

See the `Accessing_the_Guardian_API.ipynb` notebook file for details on getting the data. **Note:** This approach may be used for additional data for Assignment 2.

### Read in pre-saved data

To save time, we're loading in pre-saved data that was fetched using the Guardian API.

In [ ]:
# Load the data - articles from The Guardian
file_path = "data/"
file_name = "???_articles.json"

with open(f"{file_path}{file_name}",'r', encoding='utf-8') as fp:
    articles = json.load(fp)

print(f"Loaded {len(articles)} articles from {file_name}")

Each dictionary entry includes the *title [date]* as `key` and the *body text* from the article as `value`.

In [ ]:
article1 = list(articles.items())[0]
print("Key:",article1[?])
print("Value:",article1[?][:300],"...") # Just show first 300 characters

So the values gives us a list of documents that we can analyse.

In [ ]:
# Get a list of documents
documents = list(articles.values())

# View first 400 characters of the 1st document
documents[?][:???]

### Term Count 

**Finding important terms by the frequency of their occurance**

Using `CountVectorizer` create a `vector` for each document where the dimensionality of the vector is the `vocabulary` (all terms in the collection), and the value of each component is the number of times that the `term` occurs in the document.

All of these analyses, approach the document as a [Bag of Words](https://en.wikipedia.org/wiki/Bag-of-words_model) model. In this approach, the order of the words don't matter. A popular approach that takes into account order is [Word embedding](https://en.wikipedia.org/wiki/Word_embedding). This session does not explore word embedding.

In [ ]:
# Only count terms that in maximum of 80% of documents, and a minimum of 2 documents. 
# Count a maximum of 10000 terms, and remove common english stop words
count_vectorizer = CountVectorizer(max_df=???,min_df=?,max_features=???,stop_words="english")
count_dt_matrix = count_vectorizer.fit_transform(articles.values())

It is always important to understand the data we are working with

In [ ]:
# Take a look at the vector for the first document
doc001_vector = count_dt_matrix.toarray()[?]
doc001_vector

In [ ]:
# Get the 1000 terms identified during the vectorization process
feature_names = count_vectorizer.get_feature_names_out()
feature_names

In [ ]:
# Look at how the counts match up to the terms (for the 1st doc)
doc001_term_counts = list(zip(feature_names,doc001_vector))
doc001_term_counts

In [ ]:
# Take a look at the vocabulary which shows the total counts for whole collection
count_vectorizer.vocabulary_

#### Display matrix in dataframe

Take the term count matrix and display in a dataframe to make visible the structure


In [ ]:
# Create a new dataframe with the matrix - use titles for the index and terms for the columns
count_df = pd.DataFrame(count_dt_matrix.toarray(), columns=feature_names)
count_df

Keep a list of the titles of the articles to make it easy to see the headline that relates to the topics. We can always go back to the original documents if we need to.

In [ ]:
titles=list(articles.???)
titles

By selecting a row from the dataframe and sorting the values (counts), we can identify the top 10 terms

In [ ]:
# Sample 5 random article numbers
samples = random.sample(range(0,len(count_df)),?)
samples

In [ ]:
# View the associated terms
for sample in samples:
    doc = count_df.iloc[sample]
    title = titles[sample]
    top_terms = dict(count_df.iloc[sample].sort_values(ascending=False).head(10))
    print(f"[{sample}] {title}")
    print("\t- Top terms:",top_terms)
    print()

#### Create a top10 terms dataframe

Using the index from the documents, create a dataframe that can hold the top10 terms for each document. We also include columns for our other analysis (tfidf, lda, nmf)

In [ ]:
# Create a dataframe to hold top terms for each analysis type
terms_df = pd.DataFrame(index=count_df.index,columns=['title','count','tfidf','lda','nmf'])
terms_df['title'] = titles
terms_df

Populate the count column with data created by the count vectorizer.

In [ ]:
#For each doc, get the 10 columns with the largest counts
for idx in terms_df.index:
    counts = dict(count_df.loc[idx].sort_values(ascending=False).head(10))
    #print(counts)
    terms_df.at[idx,'count'] = list(counts.keys()) # Just the list of terms

terms_df

We should take a look at how some of the terms match up with the titles

In [ ]:
# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),5)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc['title']}")
    print("\t>> Counts:\t",doc['count'])
    print()

### Term Frequency / Inverse Document Frequency (TF/IDF)

**Finding terms that are very common in a document, but less common in the whole collection**

The [TF/IDF](https://en.wikipedia.org/wiki/Tf–idf) algorithm takes the term frequencies for a document and divides them by the frequencies of the terms in the whole collection.


In [ ]:
# Only count terms that in maximum of 80% of documents, and a minimum of 2 documents. 
# Count a maximum of 10000 terms, and remove common english stop words
tfidf_vectorizer = TfidfVectorizer(
    max_df=???, min_df=?, max_features=???, stop_words="english"
)

In [ ]:
# Get the document vectors
tfidf_dt_matrix = tfidf_vectorizer.fit_transform(articles.values())

# Display the vector for the first document
tfidf_dt_matrix.toarray()[?]

#### Display matrix in dataframe

In [ ]:
tfidf_df = pd.DataFrame(tfidf_dt_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
tfidf_df

#### Update the terms matrix

In [ ]:
for idx in terms_df.index:
    tfidf = dict(tfidf_df.loc[idx].sort_values(ascending=False).head(10))
    #print(counts)
    terms_df.at[idx,'tfidf'] = list(tfidf.keys()) 

terms_df

#### Compare approaches

In [ ]:
# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),?)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc['title']}")
    print("\t>> Counts:\t",doc['count'])
    print("\t>> TFIDF:\t",doc['tfidf'])
    print()